In [2]:
# ==========================================================
# Cell 1 — Upload ZIP (Zeek Conn AGG) + Preview head/tail  (FIXED)
# ==========================================================
import os, zipfile, glob
import pandas as pd
from google.colab import files

# 1) Upload ZIP
uploaded = files.upload()
zip_paths = [fn for fn in uploaded.keys() if fn.lower().endswith(".zip")]
if not zip_paths:
    raise SystemExit("⛔ Please upload a .zip file containing CSV(s).")

ZIP_PATH = zip_paths[0]
print("✅ Uploaded ZIP:", ZIP_PATH, "| size:", os.path.getsize(ZIP_PATH), "bytes")

# 2) Safe extract (ZIP-SLIP protected)
EXTRACT_DIR = "extracted_csvs"
os.makedirs(EXTRACT_DIR, exist_ok=True)

def safe_extract(zip_path, extract_dir):
    extract_dir_abs = os.path.abspath(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.infolist():
            # Build absolute destination path
            dest_path = os.path.abspath(os.path.join(extract_dir_abs, member.filename))
            # Ensure dest_path stays within extract_dir
            if not dest_path.startswith(extract_dir_abs + os.sep) and dest_path != extract_dir_abs:
                raise RuntimeError(f"Unsafe path in zip: {member.filename}")
        z.extractall(extract_dir_abs)

safe_extract(ZIP_PATH, EXTRACT_DIR)
print("✅ Extracted to:", EXTRACT_DIR)

# 3) Find CSVs, pick largest
csv_paths = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True) + \
            glob.glob(os.path.join(EXTRACT_DIR, "**", "*.CSV"), recursive=True)

if not csv_paths:
    raise SystemExit("⛔ No CSV files found inside the ZIP.")

csv_paths = sorted(csv_paths, key=lambda p: os.path.getsize(p), reverse=True)
CSV_PATH = csv_paths[0]

print("\n📌 Found CSVs:", len(csv_paths))
print("📌 Selected largest CSV:", CSV_PATH)
print("   size:", round(os.path.getsize(CSV_PATH)/1024/1024, 2), "MB")

# 4) Read head (safe)
print("\n🔎 Preview: HEAD(5)")
head_df = pd.read_csv(CSV_PATH, nrows=5, low_memory=False)
display(head_df)

# 5) Read tail (efficient)
print("\n🔎 Preview: TAIL(5)")
with open(CSV_PATH, "rb") as f:
    f.seek(0, os.SEEK_END)
    fsize = f.tell()
    f.seek(max(0, fsize - 20000), os.SEEK_SET)
    raw = f.read().decode("utf-8", errors="replace")

lines = [ln for ln in raw.splitlines() if ln.strip()]
tail_block = "\n".join(lines[-6:])  # header + last 5

from io import StringIO
try:
    tail_df = pd.read_csv(StringIO(tail_block), low_memory=False)
    display(tail_df.tail(5))
except Exception:
    print("⚠️ Tail parse failed, printing raw last 5 lines:\n")
    print("\n".join(lines[-5:]))

# 6) Save path for next cells
DATA_CSV = CSV_PATH
print("\n✅ DATA_CSV set to:", DATA_CSV)
print("✅ Columns:", list(head_df.columns))


Saving merged.conn.new.with.ip.zip to merged.conn.new.with.ip (1).zip
✅ Uploaded ZIP: merged.conn.new.with.ip (1).zip | size: 42676353 bytes
✅ Extracted to: extracted_csvs

📌 Found CSVs: 1
📌 Selected largest CSV: extracted_csvs/merged.conn.new.with.ip.csv
   size: 306.07 MB

🔎 Preview: HEAD(5)


,@timestamp,event.duration,network.bytes,network.packets,source.bytes,source.packets,source.port,source.ip,destination.bytes,destination.packets,...,destination.ip,zeek.connection.missed_bytes,network.transport,network.protocol,zeek.connection.state,zeek.connection.history,network.direction,zeek.connection.local_orig,zeek.connection.local_resp,zeek.connection.icmp.type
0,"Oct 16, 2025 @ 21:59:59.864","315,993,071","5,982",110,"2,664",42,"49,538",192.168.20.15,"3,318",68,...,172.16.60.2,0,tcp,http,SF,ShADTadttfF,internal,True,True,-
1,"Oct 16, 2025 @ 21:59:59.841","325,828,075","5,650",108,"2,332",40,"49,537",192.168.20.15,"3,318",68,...,172.16.60.2,0,tcp,http,SF,ShADTadttfF,internal,True,True,-
2,"Oct 16, 2025 @ 21:59:59.828","3,088,467,836","39,476",68,"30,392",42,"58,404",192.168.100.5,"9,084",26,...,192.168.100.112,0,tcp,ssl,SF,ShADTadtTFfR,internal,True,True,-
3,"Oct 16, 2025 @ 21:59:59.818","321,600,914","6,078",110,"2,760",42,"49,536",192.168.20.15,"3,318",68,...,172.16.60.2,0,tcp,http,SF,ShADTadttfF,internal,True,True,-
4,"Oct 16, 2025 @ 21:59:59.598","224,829",184,4,104,2,"10,253",192.168.100.14,80,2,...,74.178.76.128,0,tcp,-,REJ,Sr,outbound,True,False,-



🔎 Preview: TAIL(5)


,"Nov 10, 2025 @ 08:33:19.283","126,123",184,4,104,2,"57,891",192.168.20.15,80,2.1,...,173.243.143.6,0,tcp,-,REJ,Sr,outbound,True,False,-.1
0,"Nov 10, 2025 @ 08:33:19.250","36,992,073",184,4,104,2,"9,314",10.10.10.31,80,2,...,10.10.34.35,0,tcp,-,REJ,Sr,internal,True,True,-
1,"Nov 10, 2025 @ 08:33:19.207","27,898,073",184,4,104,2,"1,143",10.10.10.31,80,2,...,10.10.34.35,0,tcp,-,REJ,Sr,internal,True,True,-
2,"Nov 10, 2025 @ 08:33:18.991","3,300,031,185","19,392",56,"19,392",56,"1,900",192.168.100.251,0,0,...,239.255.255.250,0,udp,-,S0,D,outbound,True,False,-
3,"Nov 10, 2025 @ 08:33:18.928","3,007,045,984",312,6,312,6,"3,149",10.10.10.31,0,0,...,10.10.34.35,0,tcp,-,S0,S,internal,True,True,-
4,"Nov 10, 2025 @ 08:33:18.769","128,031",184,4,104,2,"57,891",192.168.20.15,80,2,...,173.243.143.6,0,tcp,-,REJ,Sr,outbound,True,False,-



✅ DATA_CSV set to: extracted_csvs/merged.conn.new.with.ip.csv
✅ Columns: ['@timestamp', 'event.duration', 'network.bytes', 'network.packets', 'source.bytes', 'source.packets', 'source.port', 'source.ip', 'destination.bytes', 'destination.packets', 'destination.port', 'destination.ip', 'zeek.connection.missed_bytes', 'network.transport', 'network.protocol', 'zeek.connection.state', 'zeek.connection.history', 'network.direction', 'zeek.connection.local_orig', 'zeek.connection.local_resp', 'zeek.connection.icmp.type']


In [3]:
# ==========================================================
# Cell 2 — Clean & Prepare df_clean (row-level)
# ==========================================================
import pandas as pd
import numpy as np

print("🔧 Loading and cleaning connection data...")

# ---- 1) Load full CSV (306MB is OK in Colab most times; if RAM issue we’ll switch to chunks)
df = pd.read_csv(DATA_CSV, low_memory=False)

print("✅ Loaded:", df.shape)
print("🧾 Columns:", list(df.columns))

# ---- 2) Helpers
def to_num(x):
    """Convert strings like '3,088,467,836' or '-' to float."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["-", "None", "nan", "NaN", ""]:
        return np.nan
    s = s.replace(",", "")
    # sometimes weird tail artifacts like '-.1' can appear; try safe cast
    try:
        return float(s)
    except:
        return np.nan

def safe_div(a, b, eps=1e-9):
    return a / (b + eps)

# ---- 3) Parse timestamp (your format is like: 'Oct 16, 2025 @ 21:59:59.864')
df["ts"] = pd.to_datetime(
    df["@timestamp"].astype(str).str.replace(" @ ", " ", regex=False),
    errors="coerce"
)

before = len(df)
df = df[df["ts"].notna()].copy()
print(f"🕒 Timestamp parsed. Dropped {before - len(df)} rows with invalid ts.")

# ---- 4) Convert numeric-like columns
num_cols = [
    "event.duration",
    "network.bytes", "network.packets",
    "source.bytes", "source.packets", "source.port",
    "destination.bytes", "destination.packets", "destination.port",
    "zeek.connection.missed_bytes",
    "zeek.connection.icmp.type",
]
for c in num_cols:
    if c in df.columns:
        df[c] = df[c].map(to_num)

# ---- 5) Duration to seconds
# In Zeek/ECS, event.duration is often nanoseconds. We'll treat it as ns if values look huge.
# If it's already ms/s, we can adjust later; for now: duration_s = duration_ns / 1e9
df["duration_s"] = df["event.duration"] / 1e9
df["duration_s"] = df["duration_s"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

# clip extremely small to avoid divide-by-zero explosions
df["duration_s_clip"] = df["duration_s"].clip(lower=0.001)

# ---- 6) Row-level traffic features (useful later for aggregation too)
df["bps"] = safe_div(df["network.bytes"].fillna(0.0), df["duration_s_clip"])
df["pps"] = safe_div(df["network.packets"].fillna(0.0), df["duration_s_clip"])
df["bpp"] = safe_div(df["network.bytes"].fillna(0.0), df["network.packets"].fillna(0.0))

df["src_to_dst_byte_ratio"] = safe_div(df["source.bytes"].fillna(0.0), df["destination.bytes"].fillna(0.0))
df["src_to_dst_pkt_ratio"]  = safe_div(df["source.packets"].fillna(0.0), df["destination.packets"].fillna(0.0))

# ---- 7) Normalize some categorical columns (keep as string/category)
cat_cols = [
    "network.transport", "network.protocol",
    "zeek.connection.state", "zeek.connection.history",
    "network.direction",
    "zeek.connection.local_orig", "zeek.connection.local_resp",
]
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype(str)

# ---- 8) Sort and keep df_clean
df_clean = df.sort_values("ts").reset_index(drop=True)

print("✅ df_clean ready:", df_clean.shape)
print("🕒 Time range:", df_clean["ts"].min(), "→", df_clean["ts"].max())

display(df_clean[[
    "ts","source.ip","destination.ip","network.direction",
    "network.bytes","network.packets","duration_s","bps","pps","bpp",
    "destination.port","network.transport","network.protocol","zeek.connection.state"
]].head(5))

display(df_clean[[
    "ts","source.ip","destination.ip","network.direction",
    "network.bytes","network.packets","duration_s","bps","pps","bpp",
    "destination.port","network.transport","network.protocol","zeek.connection.state"
]].tail(5))


🔧 Loading and cleaning connection data...
✅ Loaded: (2169590, 21)
🧾 Columns: ['@timestamp', 'event.duration', 'network.bytes', 'network.packets', 'source.bytes', 'source.packets', 'source.port', 'source.ip', 'destination.bytes', 'destination.packets', 'destination.port', 'destination.ip', 'zeek.connection.missed_bytes', 'network.transport', 'network.protocol', 'zeek.connection.state', 'zeek.connection.history', 'network.direction', 'zeek.connection.local_orig', 'zeek.connection.local_resp', 'zeek.connection.icmp.type']
🕒 Timestamp parsed. Dropped 0 rows with invalid ts.
✅ df_clean ready: (2169590, 29)
🕒 Time range: 2025-10-16 20:13:49.580000 → 2025-11-10 11:34:48.426000


,ts,source.ip,destination.ip,network.direction,network.bytes,network.packets,duration_s,bps,pps,bpp,destination.port,network.transport,network.protocol,zeek.connection.state
0,2025-10-16 20:13:49.580,192.168.100.14,146.75.122.172,outbound,184.0,4.0,0.000054,183999.816,3999.996,46.0,80.0,tcp,-,REJ
1,2025-10-16 20:13:49.681,192.168.100.5,20.44.239.154,outbound,184.0,4.0,0.000051,183999.816,3999.996,46.0,443.0,tcp,-,REJ
2,2025-10-16 20:13:49.691,192.168.100.5,20.44.239.154,outbound,184.0,4.0,0.000110,183999.816,3999.996,46.0,443.0,tcp,-,REJ
3,2025-10-16 20:13:49.704,192.168.100.31,192.168.100.5,internal,378.0,4.0,0.000489,377999.622,3999.996,94.5,53.0,udp,dns,SF
4,2025-10-16 20:13:49.704,192.168.100.31,192.168.100.5,internal,308.0,4.0,0.000479,307999.692,3999.996,77.0,53.0,udp,dns,SF


,ts,source.ip,destination.ip,network.direction,network.bytes,network.packets,duration_s,bps,pps,bpp,destination.port,network.transport,network.protocol,zeek.connection.state
2169585,2025-11-10 11:34:45.475,192.168.20.13,192.168.100.33,internal,104.0,2.0,0.000005,103999.896,1999.998,52.0,8383.0,tcp,-,S0
2169586,2025-11-10 11:34:45.646,192.168.20.13,192.168.100.33,internal,104.0,2.0,0.000006,103999.896,1999.998,52.0,8383.0,tcp,-,S0
2169587,2025-11-10 11:34:45.848,192.168.20.13,192.168.100.33,internal,104.0,2.0,0.000006,103999.896,1999.998,52.0,8383.0,tcp,-,S0
2169588,2025-11-10 11:34:45.999,192.168.20.13,192.168.100.33,internal,104.0,2.0,0.000007,103999.896,1999.998,52.0,8383.0,tcp,-,S0
2169589,2025-11-10 11:34:48.426,192.168.20.15,37.32.10.75,outbound,184.0,4.0,0.000443,183999.816,3999.996,46.0,21116.0,tcp,-,REJ


In [9]:
# ==========================================================
# Cell 3 — Feature Engineering ONLY (NO ML, NO severity)
# Outputs:
#   - agg_host          : (source.ip, window_start) features for slow/volume/exfil
#   - agg_beacon_host   : (source.ip, window_start) beaconing features (built from src-dst)
#   - agg_features      : merged feature table (host×minute)
# ==========================================================
import numpy as np
import pandas as pd

# -------------------------------
# 0) Start from df_clean
# -------------------------------
df = df_clean.copy()

# Remove noisy IPs: IPv6 + 0.0.0.0
df = df[~df["source.ip"].astype(str).str.contains(":", regex=False)]
df = df[df["source.ip"] != "0.0.0.0"].copy()
print("✅ Rows after IP filtering:", len(df))

# Window: 1 minute
df["window_start"] = df["ts"].dt.floor("1min")

# Ensure numeric columns
num_force = [
    "source.bytes", "destination.bytes",
    "network.bytes", "network.packets",
    "bps", "pps", "bpp",
    "destination.port",
    "zeek.connection.missed_bytes"
]
for c in num_force:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

# -------------------------------
# 1) Robust outbound/inbound per row
# -------------------------------
lo = df["zeek.connection.local_orig"].astype(str).str.lower().isin(["true", "1", "yes"])
lr = df["zeek.connection.local_resp"].astype(str).str.lower().isin(["true", "1", "yes"])

df["outbound_bytes"] = 0.0
df["inbound_bytes"] = 0.0

mask_out = lo & (~lr)     # local -> remote
mask_in  = (~lo) & lr     # remote -> local
amb      = ~(mask_out | mask_in)

df.loc[mask_out, "outbound_bytes"] = df.loc[mask_out, "source.bytes"]
df.loc[mask_out, "inbound_bytes"]  = df.loc[mask_out, "destination.bytes"]

df.loc[mask_in, "outbound_bytes"] = df.loc[mask_in, "destination.bytes"]
df.loc[mask_in, "inbound_bytes"]  = df.loc[mask_in, "source.bytes"]

# Fallback for ambiguous rows using direction
is_out_dir = df.loc[amb, "network.direction"].astype(str).str.lower() == "outbound"
df.loc[amb & is_out_dir, "outbound_bytes"] = df.loc[amb & is_out_dir, "network.bytes"]
df.loc[amb & is_out_dir, "inbound_bytes"]  = 0.0
df.loc[amb & (~is_out_dir), "inbound_bytes"] = df.loc[amb & (~is_out_dir), "network.bytes"]
df.loc[amb & (~is_out_dir), "outbound_bytes"] = 0.0

# -------------------------------
# 2) Success / failure flags + success+reply gate
# -------------------------------
state = df["zeek.connection.state"].astype(str).str.upper().fillna("")
df["is_success"] = (state == "SF").astype(int)     # conservative
df["is_reject"]  = (state == "REJ").astype(int)
df["is_fail"]    = ((state == "S0") | (state == "REJ")).astype(int)

df["has_reply_data"] = ((df["inbound_bytes"] > 0) | (df["destination.bytes"] > 0)).astype(int)
df["is_success_reply"] = ((df["is_success"] == 1) & (df["has_reply_data"] == 1)).astype(int)

df["outbound_bytes_sr"] = df["outbound_bytes"] * df["is_success_reply"]
df["inbound_bytes_sr"]  = df["inbound_bytes"]  * df["is_success_reply"]

# ==========================================================
# 3) Host×Minute Aggregation (Slow/Exfil base features)
# ==========================================================
g_host = ["source.ip", "window_start"]

agg_host = df.groupby(g_host).agg(
    conn_count=("ts", "size"),

    # diversity
    uniq_dst_ips=("destination.ip", pd.Series.nunique),
    uniq_dst_ports=("destination.port", pd.Series.nunique),

    # totals (all)
    bytes_sum=("network.bytes", "sum"),
    pkts_sum=("network.packets", "sum"),

    outbound_bytes=("outbound_bytes", "sum"),
    inbound_bytes=("inbound_bytes", "sum"),

    # totals (success+reply only)
    outbound_bytes_sr=("outbound_bytes_sr", "sum"),
    inbound_bytes_sr=("inbound_bytes_sr", "sum"),

    # quality counters
    success_conn_count=("is_success", "sum"),
    reject_conn_count=("is_reject", "sum"),
    fail_conn_count=("is_fail", "sum"),

    missed_bytes_sum=("zeek.connection.missed_bytes", "sum"),

    # row-level rates (averages)
    bps_mean=("bps", "mean"),
    pps_mean=("pps", "mean"),
    bpp_mean=("bpp", "mean"),
).reset_index()

eps = 1e-6
agg_host["success_rate"] = agg_host["success_conn_count"] / (agg_host["conn_count"] + eps)
agg_host["fail_rate"]    = agg_host["fail_conn_count"]    / (agg_host["conn_count"] + eps)

# stable dominance metrics (both all-traffic and success+reply)
agg_host["log_out_minus_in"] = np.log1p(agg_host["outbound_bytes"]) - np.log1p(agg_host["inbound_bytes"])
agg_host["log_out_minus_in_sr"] = np.log1p(agg_host["outbound_bytes_sr"]) - np.log1p(agg_host["inbound_bytes_sr"])

# rolling/trend for success+reply outbound (per host)
agg_host = agg_host.sort_values(["source.ip", "window_start"]).reset_index(drop=True)

def add_slow_features(g):
    bout = g["outbound_bytes_sr"].values
    g["outbound_sr_roll_60m"] = pd.Series(bout).rolling(60, min_periods=1).sum().values

    roll30 = pd.Series(bout).rolling(30, min_periods=1).sum().values
    slope = np.zeros_like(roll30)
    if len(roll30) > 30:
        slope[30:] = roll30[30:] - roll30[:-30]
    g["outbound_sr_trend_30m"] = slope
    return g

agg_host = agg_host.groupby("source.ip", group_keys=False).apply(add_slow_features)

print("✅ agg_host:", agg_host.shape)
display(agg_host.head(3))

# ==========================================================
# 4) Beaconing Features via (src,dst,minute) then reduce to host×minute
# ==========================================================
g_pair = ["source.ip", "destination.ip", "window_start"]

pair = df.groupby(g_pair).agg(
    pair_conn_count=("ts", "size"),
).reset_index()

pair = pair.sort_values(["source.ip", "destination.ip", "window_start"]).reset_index(drop=True)

def add_beacon_pair(g):
    x = g["pair_conn_count"].values
    active = (x > 0).astype(int)

    g["persist_5m_pair"] = pd.Series(active).rolling(5, min_periods=1).sum().values

    t = g["window_start"].values.astype("datetime64[ns]")
    idx = np.where(active == 1)[0]
    if len(idx) >= 3:
        dt = np.diff(t[idx]).astype("timedelta64[s]").astype(float) / 60.0
        cv = np.std(dt) / (np.mean(dt) + 1e-9)
        reg = 1.0 / (cv + 1e-6)
    else:
        reg = 0.0

    g["beacon_reg_pair"] = reg
    return g

pair = pair.groupby(["source.ip", "destination.ip"], group_keys=False).apply(add_beacon_pair)

# raw beacon score per pair (no scoring policy, just a feature)
pair["beacon_score_pair"] = (
    np.log1p(pair["beacon_reg_pair"]) * 0.65 +
    (pair["persist_5m_pair"] / 5.0) * 0.35
)

# Reduce to host×minute: take max beacon score across destinations + keep best dst
pair_sorted = pair.sort_values(
    ["source.ip", "window_start", "beacon_score_pair"],
    ascending=[True, True, False]
)

agg_beacon_host = pair_sorted.groupby(["source.ip", "window_start"]).agg(
    beacon_score=("beacon_score_pair", "first"),
    beacon_best_dst=("destination.ip", "first"),
    beacon_best_pair_conn_count=("pair_conn_count", "first"),
    beacon_best_persist_5m=("persist_5m_pair", "first"),
    beacon_best_regularity=("beacon_reg_pair", "first"),
).reset_index()

print("✅ agg_beacon_host:", agg_beacon_host.shape)
display(agg_beacon_host.head(3))

# ==========================================================
# 5) Merge feature tables => final feature-only dataset
# ==========================================================
agg_features = agg_host.merge(
    agg_beacon_host,
    on=["source.ip", "window_start"],
    how="left"
)

# fill beacon NaNs (host-minute with no dst-pairs in pair table)
for c in ["beacon_score", "beacon_best_pair_conn_count", "beacon_best_persist_5m", "beacon_best_regularity"]:
    agg_features[c] = agg_features[c].fillna(0.0)
agg_features["beacon_best_dst"] = agg_features["beacon_best_dst"].fillna("")

print("✅ agg_features (feature-only):", agg_features.shape)
display(agg_features.head(5))
display(agg_features.tail(5))

# Optional: save features for downstream ML cells (no scoring)
FEATURES_CSV = "conn_agg_features_only.csv"
agg_features.to_csv(FEATURES_CSV, index=False)
print("📦 Saved feature-only CSV:", FEATURES_CSV)


✅ Rows after IP filtering: 2160404
✅ agg_host: (92862, 24)


/tmp/ipython-input-2014657011.py:134: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_host = agg_host.groupby("source.ip", group_keys=False).apply(add_slow_features)


,source.ip,window_start,conn_count,uniq_dst_ips,uniq_dst_ports,bytes_sum,pkts_sum,outbound_bytes,inbound_bytes,outbound_bytes_sr,...,missed_bytes_sum,bps_mean,pps_mean,bpp_mean,success_rate,fail_rate,log_out_minus_in,log_out_minus_in_sr,outbound_sr_roll_60m,outbound_sr_trend_30m
0,10.10.10.1,2025-10-18 11:50:00,1,1,1,224.0,4.0,0.0,224.0,0.0,...,0.0,48379.189173,863.914092,56.000000,0.0,0.000000,-5.416100,0.0,0.0,0.0
1,10.10.10.1,2025-10-22 11:27:00,1,1,1,224.0,4.0,0.0,224.0,0.0,...,0.0,34524.819800,616.514639,56.000000,0.0,0.000000,-5.416100,0.0,0.0,0.0
2,10.10.10.1,2025-10-24 13:46:00,1,1,1,55835.0,351.0,0.0,55835.0,0.0,...,0.0,44.312653,0.278566,159.074074,0.0,0.999999,-10.930174,0.0,0.0,0.0


/tmp/ipython-input-2014657011.py:168: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pair = pair.groupby(["source.ip", "destination.ip"], group_keys=False).apply(add_beacon_pair)


✅ agg_beacon_host: (92862, 7)


,source.ip,window_start,beacon_score,beacon_best_dst,beacon_best_pair_conn_count,beacon_best_persist_5m,beacon_best_regularity
0,10.10.10.1,2025-10-18 11:50:00,0.070000,10.10.10.51,1,1.0,0.000000
1,10.10.10.1,2025-10-22 11:27:00,0.070000,10.10.10.35,1,1.0,0.000000
2,10.10.10.1,2025-10-24 13:46:00,0.197806,255.255.255.255,1,1.0,0.217287


✅ agg_features (feature-only): (92862, 29)


,source.ip,window_start,conn_count,uniq_dst_ips,uniq_dst_ports,bytes_sum,pkts_sum,outbound_bytes,inbound_bytes,outbound_bytes_sr,...,fail_rate,log_out_minus_in,log_out_minus_in_sr,outbound_sr_roll_60m,outbound_sr_trend_30m,beacon_score,beacon_best_dst,beacon_best_pair_conn_count,beacon_best_persist_5m,beacon_best_regularity
0,10.10.10.1,2025-10-18 11:50:00,1,1,1,224.0,4.0,0.0,224.0,0.0,...,0.000000,-5.416100,0.0,0.0,0.0,0.070000,10.10.10.51,1,1.0,0.000000
1,10.10.10.1,2025-10-22 11:27:00,1,1,1,224.0,4.0,0.0,224.0,0.0,...,0.000000,-5.416100,0.0,0.0,0.0,0.070000,10.10.10.35,1,1.0,0.000000
2,10.10.10.1,2025-10-24 13:46:00,1,1,1,55835.0,351.0,0.0,55835.0,0.0,...,0.999999,-10.930174,0.0,0.0,0.0,0.197806,255.255.255.255,1,1.0,0.217287
3,10.10.10.1,2025-10-24 14:08:00,1,1,1,1162.0,7.0,0.0,1162.0,0.0,...,0.999999,-7.058758,0.0,0.0,0.0,0.267806,255.255.255.255,1,2.0,0.217287
4,10.10.10.1,2025-10-24 14:10:00,1,1,1,74000.0,465.0,0.0,74000.0,0.0,...,0.999999,-11.211834,0.0,0.0,0.0,0.337806,255.255.255.255,1,3.0,0.217287


,source.ip,window_start,conn_count,uniq_dst_ips,uniq_dst_ports,bytes_sum,pkts_sum,outbound_bytes,inbound_bytes,outbound_bytes_sr,...,fail_rate,log_out_minus_in,log_out_minus_in_sr,outbound_sr_roll_60m,outbound_sr_trend_30m,beacon_score,beacon_best_dst,beacon_best_pair_conn_count,beacon_best_persist_5m,beacon_best_regularity
92857,94.101.185.206,2025-11-07 16:26:00,1,1,1,80.0,2.0,0.0,80.0,0.0,...,0.0,-4.394449,0.0,0.0,0.0,0.546374,192.168.20.15,1,5.0,0.352716
92858,94.101.185.206,2025-11-09 16:11:00,1,1,1,80.0,2.0,0.0,80.0,0.0,...,0.0,-4.394449,0.0,0.0,0.0,0.546374,192.168.20.15,1,5.0,0.352716
92859,94.101.185.206,2025-11-09 16:49:00,1,1,1,80.0,2.0,0.0,80.0,0.0,...,0.0,-4.394449,0.0,0.0,0.0,0.546374,192.168.20.15,1,5.0,0.352716
92860,94.101.185.206,2025-11-09 17:40:00,1,1,1,80.0,2.0,0.0,80.0,0.0,...,0.0,-4.394449,0.0,0.0,0.0,0.546374,192.168.20.15,1,5.0,0.352716
92861,94.101.185.206,2025-11-09 20:29:00,1,1,1,80.0,2.0,0.0,80.0,0.0,...,0.0,-4.394449,0.0,0.0,0.0,0.546374,192.168.20.15,1,5.0,0.352716


📦 Saved feature-only CSV: conn_agg_features_only.csv


In [10]:
# ==========================================================
# Cell 4 — Autoencoder (AE) anomaly score on agg_features (unsupervised)
# Output:
#   agg_scored: agg_features + ae_score
# ==========================================================
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models

# ---- 0) Base table from Cell 3
df_feat = agg_features.copy()

# ---- 1) Choose numeric feature columns for AE
# (Exclude identifiers + string cols)
exclude_cols = {"source.ip", "window_start", "beacon_best_dst"}
num_cols = [c for c in df_feat.columns if c not in exclude_cols]

# keep only numeric columns
num_cols = [c for c in num_cols if pd.api.types.is_numeric_dtype(df_feat[c])]

# replace inf/nan
X = df_feat[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype(np.float32)

# ---- 2) Time-based split (train on early period)
df_feat = df_feat.sort_values("window_start").reset_index(drop=True)
X = df_feat[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).values.astype(np.float32)

split = int(len(df_feat) * 0.7)
X_train = X[:split]
X_all = X

# ---- 3) Scale
scaler_ae = StandardScaler()
X_train_s = scaler_ae.fit_transform(X_train)
X_all_s = scaler_ae.transform(X_all)

input_dim = X_all_s.shape[1]
encoding_dim = max(8, min(32, input_dim // 2))

# ---- 4) Build AE
inp = layers.Input(shape=(input_dim,))
x = layers.Dense(encoding_dim, activation="relu")(inp)
x = layers.Dense(max(4, encoding_dim//2), activation="relu")(x)
x = layers.Dense(encoding_dim, activation="relu")(x)
out = layers.Dense(input_dim, activation="linear")(x)

ae = models.Model(inp, out)
ae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

# ---- 5) Train
cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
]

ae.fit(
    X_train_s, X_train_s,
    validation_split=0.1,
    epochs=30,
    batch_size=512,
    verbose=1,
    callbacks=cb
)

# ---- 6) Reconstruction error => ae_score
X_hat = ae.predict(X_all_s, batch_size=1024, verbose=0)
recon_mse = np.mean((X_all_s - X_hat) ** 2, axis=1)

df_feat["ae_score"] = recon_mse

# Save intermediate
agg_scored = df_feat.copy()

print("✅ AE done. ae_score stats:")
print(agg_scored["ae_score"].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.995]))
display(agg_scored[["source.ip","window_start","ae_score"]].sort_values("ae_score", ascending=False).head(10))


Epoch 1/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 1.0233 - val_loss: 1.2191
Epoch 2/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7194 - val_loss: 1.0180
Epoch 3/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5525 - val_loss: 0.8930
Epoch 4/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5031 - val_loss: 0.8112
Epoch 5/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5475 - val_loss: 0.7390
Epoch 6/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3038 - val_loss: 0.7010
Epoch 7/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3253 - val_loss: 0.6760
Epoch 8/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2475 - val_loss: 0.6656
Epoch 9/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2259 - val_loss: 0.6464
Epoch 10/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2248 - val_loss: 0.6329
Epoch 11/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2086 - val_loss: 0.6189
Epoch 12/30
115/115 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

,source.ip,window_start,ae_score
58626,192.168.20.15,2025-11-04 11:50:00,557.492065
59256,192.168.10.13,2025-11-04 12:05:00,422.207520
72107,192.168.100.28,2025-11-07 12:32:00,351.800446
59276,192.168.10.11,2025-11-04 12:05:00,245.466400
60221,192.168.20.33,2025-11-04 12:27:00,222.114716
89424,10.10.10.31,2025-11-10 10:09:00,208.278061
59981,192.168.20.33,2025-11-04 12:21:00,163.812424
59918,192.168.20.13,2025-11-04 12:20:00,108.590141
50799,192.168.20.10,2025-11-01 12:50:00,107.729736
7099,10.10.10.31,2025-10-18 11:20:00,93.355064


In [11]:
# ==========================================================
# Cell 5 — Sequence model (GRU/LSTM) prediction error per host-minute
# Output:
#   agg_scored: + lstm_score
# ==========================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models

df_seq = agg_scored.copy()
df_seq = df_seq.sort_values(["source.ip","window_start"]).reset_index(drop=True)

# ---- 1) Select a SMALL set of stable features for sequence prediction
# (keep it compact + meaningful for beacon & slow exfil)
seq_cols = [
    "conn_count",
    "uniq_dst_ips",
    "uniq_dst_ports",
    "outbound_sr_roll_60m",
    "outbound_sr_trend_30m",
    "success_rate",
    "fail_rate",
    "beacon_score",
    "log_out_minus_in_sr",
]
# keep only existing numeric cols
seq_cols = [c for c in seq_cols if c in df_seq.columns and pd.api.types.is_numeric_dtype(df_seq[c])]

# log-transform heavy tails
for c in ["conn_count","uniq_dst_ips","uniq_dst_ports","outbound_sr_roll_60m"]:
    if c in df_seq.columns:
        df_seq[c] = np.log1p(df_seq[c].astype(float))

# fill nan/inf
df_seq[seq_cols] = df_seq[seq_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

# ---- 2) Scale globally (works fine for unsupervised sequence)
scaler_seq = StandardScaler()
df_seq[seq_cols] = scaler_seq.fit_transform(df_seq[seq_cols].values.astype(np.float32))

# ---- 3) Build sequences: window W predicts next step
W = 20  # lookback minutes
MIN_LEN = W + 1

X_list, y_list, idx_list = [], [], []

# To keep it deployable, only build sequences for hosts with enough history
for ip, g in df_seq.groupby("source.ip", sort=False):
    if len(g) < MIN_LEN:
        continue
    arr = g[seq_cols].values.astype(np.float32)
    # sliding windows
    for i in range(W, len(arr)):
        X_list.append(arr[i-W:i])
        y_list.append(arr[i])           # predict next vector
        idx_list.append(g.index[i])     # align score to time i

X_seq = np.stack(X_list, axis=0)
y_seq = np.stack(y_list, axis=0)
idx_seq = np.array(idx_list)

print("✅ Sequences built:", X_seq.shape, "Targets:", y_seq.shape)

# ---- 4) Time-based split on the sequence samples (by window_start order)
# Use indices already mapped to df_seq ordering
order = np.argsort(idx_seq)
X_seq = X_seq[order]
y_seq = y_seq[order]
idx_seq = idx_seq[order]

split = int(len(X_seq) * 0.7)
X_train, y_train = X_seq[:split], y_seq[:split]
X_val, y_val     = X_seq[split:split+int(len(X_seq)*0.1)], y_seq[split:split+int(len(X_seq)*0.1)]

# ---- 5) Build GRU model (swap GRU->LSTM if you want)
inp = layers.Input(shape=(W, len(seq_cols)))
x = layers.GRU(32, return_sequences=False)(inp)
x = layers.Dense(32, activation="relu")(x)
out = layers.Dense(len(seq_cols), activation="linear")(x)

seq_model = models.Model(inp, out)
seq_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

cb = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
]

seq_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=512,
    verbose=1,
    callbacks=cb
)

# ---- 6) Prediction error => lstm_score (for rows that have sequences)
y_hat = seq_model.predict(X_seq, batch_size=1024, verbose=0)
pred_mse = np.mean((y_seq - y_hat) ** 2, axis=1)

df_seq["lstm_score"] = 0.0
df_seq.loc[idx_seq, "lstm_score"] = pred_mse

# Save back
agg_scored = df_seq.copy()

print("✅ Sequence model done. lstm_score stats (non-zero rows):")
non0 = agg_scored.loc[agg_scored["lstm_score"] > 0, "lstm_score"]
print(non0.describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.995]))
display(agg_scored[["source.ip","window_start","lstm_score"]].sort_values("lstm_score", ascending=False).head(10))


✅ Sequences built: (90827, 20, 9) Targets: (90827, 9)
Epoch 1/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - loss: 0.6860 - val_loss: 0.3116
Epoch 2/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - loss: 0.2848 - val_loss: 0.2205
Epoch 3/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - loss: 0.2479 - val_loss: 0.1879
Epoch 4/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - loss: 0.2252 - val_loss: 0.1740
Epoch 5/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - loss: 0.2219 - val_loss: 0.1704
Epoch 6/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 0.2158 - val_loss: 0.1682
Epoch 7/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - loss: 0.2045 - val_loss: 0.1663
Epoch 8/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - loss: 0.2042 - val_loss: 0.1655
Epoch 9/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - loss: 0.2024 - val_loss: 0.1686
Epoch 10/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - loss: 0.1983 - val_loss: 0.1633
Epoch 11/25
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - loss: 0.1984 - va

,source.ip,window_start,lstm_score
2537,10.10.10.31,2025-10-18 11:51:00,223.016846
2507,10.10.10.31,2025-10-18 11:20:00,69.185104
72774,192.168.20.10,2025-10-26 13:50:00,65.826469
2567,10.10.10.31,2025-10-19 14:15:00,41.097672
16372,192.168.10.15,2025-11-02 12:27:00,21.580244
253,10.10.10.18,2025-11-10 09:51:00,21.141642
72744,192.168.20.10,2025-10-26 13:20:00,19.840752
1718,10.10.10.200,2025-11-10 09:57:00,19.415586
2559,10.10.10.31,2025-10-19 14:07:00,16.314331
175,10.10.10.18,2025-11-06 10:14:00,16.235489


In [13]:
# ==========================================================
# Cell 6 — Fuse AE + LSTM scores, enforce 0.5% policy, build severity 0..99, export CSV
# ==========================================================
import numpy as np
import pandas as pd

df_out = agg_scored.copy()

# safety
df_out["ae_score"] = pd.to_numeric(df_out.get("ae_score", 0.0), errors="coerce").fillna(0.0)
df_out["lstm_score"] = pd.to_numeric(df_out.get("lstm_score", 0.0), errors="coerce").fillna(0.0)

n = len(df_out)

# Percentile ranks (stable)
df_out["ae_pct"]   = (df_out["ae_score"].rank(method="average") - 1) / (n - 1 + 1e-9)
df_out["lstm_pct"] = (df_out["lstm_score"].rank(method="average") - 1) / (n - 1 + 1e-9)

df_out["fused_pct"] = np.maximum(df_out["ae_pct"], df_out["lstm_pct"])

# Enforce top 0.5%
RATE = 0.005
k = int(np.ceil(n * RATE))

top_idx = df_out["fused_pct"].nlargest(k).index
df_out["is_alert"] = 0
df_out.loc[top_idx, "is_alert"] = 1

# Severity scaling inside tail
df_out["severity"] = 0
tail = df_out.loc[df_out["is_alert"] == 1, "fused_pct"].values
tail_norm = (tail - tail.min()) / (tail.max() - tail.min() + 1e-12)

gamma = 3.0
sev_tail = 1 + np.floor((tail_norm ** (1.0/gamma)) * 98).astype(int)
df_out.loc[df_out["is_alert"] == 1, "severity"] = np.clip(sev_tail, 1, 99)

# Explainable alert type (heuristic labeling, NOT a separate detector)
# Use percentiles of beacon_score and outbound_sr_roll_60m to label within alerts
if "beacon_score" in df_out.columns:
    df_out["beacon_pct_feat"] = (df_out["beacon_score"].rank(method="average") - 1) / (n - 1 + 1e-9)
else:
    df_out["beacon_pct_feat"] = 0.0

if "outbound_sr_roll_60m" in df_out.columns:
    tmp = df_out["outbound_sr_roll_60m"].replace([np.inf,-np.inf], np.nan).fillna(0.0).astype(float)
    df_out["slow_pct_feat"] = (tmp.rank(method="average") - 1) / (n - 1 + 1e-9)
else:
    df_out["slow_pct_feat"] = 0.0

df_out["alert_type"] = ""
mask = df_out["severity"] > 0
df_out.loc[mask, "alert_type"] = np.where(
    df_out.loc[mask, "beacon_pct_feat"] >= df_out.loc[mask, "slow_pct_feat"],
    "BEACONING",
    "SLOW_EXFIL"
)

print("✅ Total rows:", n)
print("✅ severity>0 rows:", int((df_out["severity"]>0).sum()),
      f"({(df_out['severity']>0).mean()*100:.3f}%) target=0.5%")

print("✅ Split (severity>0):")
display(df_out.loc[df_out["severity"]>0, "alert_type"].value_counts())

display(
    df_out[[
        "source.ip","window_start",
        "ae_score","lstm_score","ae_pct","lstm_pct","fused_pct",
        "severity","alert_type","beacon_best_dst"
    ]].sort_values("severity", ascending=False).head(30)
)

OUT_CSV = "conn_ml_scored_severity.csv"
df_out.to_csv(OUT_CSV, index=False)
print("📦 Saved:", OUT_CSV)

from google.colab import files
files.download(OUT_CSV)


✅ Total rows: 92862
✅ severity>0 rows: 465 (0.501%) target=0.5%
✅ Split (severity>0):


,count
alert_type,
BEACONING,305
SLOW_EXFIL,160


,source.ip,window_start,ae_score,lstm_score,ae_pct,lstm_pct,fused_pct,severity,alert_type,beacon_best_dst
84498,192.168.20.33,2025-11-04 12:21:00,163.812424,2.932275,0.999935,0.992774,0.999935,98,BEACONING,23.46.189.152
76901,192.168.20.13,2025-11-04 12:20:00,108.590141,2.294149,0.999925,0.991816,0.999925,98,BEACONING,216.58.206.35
13910,192.168.10.13,2025-11-04 12:05:00,422.207520,1.089073,0.999989,0.977719,0.999989,98,BEACONING,95.217.49.192
2567,10.10.10.31,2025-10-19 14:15:00,1.934578,41.097672,0.991547,0.999968,0.999968,98,SLOW_EXFIL,3.233.158.25
45927,192.168.100.28,2025-11-07 12:32:00,351.800446,2.948296,0.999978,0.992806,0.999978,98,BEACONING,199.232.210.172
11270,192.168.10.11,2025-11-04 12:05:00,245.466400,1.160573,0.999968,0.980304,0.999968,98,SLOW_EXFIL,52.123.129.14
2922,10.10.10.31,2025-11-10 10:09:00,208.278061,1.086043,0.999946,0.977558,0.999946,98,SLOW_EXFIL,185.143.233.5
1718,10.10.10.200,2025-11-10 09:57:00,2.116078,19.415586,0.992343,0.999925,0.999925,98,SLOW_EXFIL,10.10.10.1
253,10.10.10.18,2025-11-10 09:51:00,5.615016,21.141642,0.998040,0.999946,0.999946,98,BEACONING,157.240.30.142
79728,192.168.20.15,2025-11-04 11:50:00,557.492065,0.662395,1.000000,0.936916,1.000000,98,BEACONING,172.217.18.14


📦 Saved: conn_ml_scored_severity.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# ==========================================================
# Cell 7 — Alert Context Enrichment (Explainability)
# Adds per-alert context fields from raw df to final df_out
# Output:
#   conn_ml_scored_severity_enriched.csv
# ==========================================================
import numpy as np
import pandas as pd

# ---- 0) Safety: required frames
if "df" not in globals():
    raise SystemExit("⛔ df not found. Run Cell 3 (feature-only) first.")
if "df_out" not in globals():
    raise SystemExit("⛔ df_out not found. Run Cell 6 first (final scoring).")

raw = df.copy()
out = df_out.copy()

# Ensure types match for merge keys
raw["window_start"] = pd.to_datetime(raw["window_start"])
out["window_start"] = pd.to_datetime(out["window_start"])

# ---- 1) Focus only on alerts (fast)
alerts = out.loc[out["severity"] > 0, ["source.ip", "window_start"]].drop_duplicates()
print("✅ Alerts to enrich:", len(alerts))

# Merge to get raw rows for alert windows
raw_alert = raw.merge(alerts, on=["source.ip", "window_start"], how="inner")
print("✅ Raw rows for alerts:", len(raw_alert))

# ---- 2) Helper: get top-N summary per alert-window
def top_value(s):
    # return the most frequent non-null value
    s = s.dropna()
    if len(s) == 0:
        return ""
    return s.value_counts().idxmax()

def top_value_numeric(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return float(s.value_counts().idxmax())

# Ensure numeric bytes/packets exist
for c in ["network.bytes", "network.packets", "outbound_bytes", "inbound_bytes", "outbound_bytes_sr", "inbound_bytes_sr"]:
    if c in raw_alert.columns:
        raw_alert[c] = pd.to_numeric(raw_alert[c], errors="coerce").fillna(0.0)

# ---- 3) Aggregate raw context per (source.ip, window_start)
ctx = raw_alert.groupby(["source.ip", "window_start"]).agg(
    raw_rows=("ts", "size"),

    # destination summary
    top_dst_ip=("destination.ip", top_value),
    top_dst_port=("destination.port", top_value_numeric),
    top_transport=("network.transport", top_value),
    top_protocol=("network.protocol", top_value),

    # volume summary (window level)
    raw_bytes_sum=("network.bytes", "sum"),
    raw_pkts_sum=("network.packets", "sum"),

    # outbound/inbound summary (all traffic)
    raw_out_bytes_sum=("outbound_bytes", "sum"),
    raw_in_bytes_sum=("inbound_bytes", "sum"),

    # gated success+reply summary (if present)
    raw_out_sr_sum=("outbound_bytes_sr", "sum") if "outbound_bytes_sr" in raw_alert.columns else ("network.bytes", "sum"),
    raw_in_sr_sum=("inbound_bytes_sr", "sum") if "inbound_bytes_sr" in raw_alert.columns else ("network.bytes", "sum"),

    # quality summary
    sf_count=("is_success", "sum") if "is_success" in raw_alert.columns else ("ts", "size"),
    rej_count=("is_reject", "sum") if "is_reject" in raw_alert.columns else ("ts", "size"),
    fail_count=("is_fail", "sum") if "is_fail" in raw_alert.columns else ("ts", "size"),
).reset_index()

# Derived context ratios
eps = 1e-6
ctx["raw_success_rate"] = ctx["sf_count"] / (ctx["raw_rows"] + eps)
ctx["raw_fail_rate"]    = ctx["fail_count"] / (ctx["raw_rows"] + eps)
ctx["raw_out_in_ratio"] = ctx["raw_out_bytes_sum"] / (ctx["raw_in_bytes_sum"] + eps)

# ---- 4) Add a compact "reason" string (human readable)
def build_reason(r):
    parts = []
    if r.get("top_dst_ip", ""):
        parts.append(f"top_dst={r['top_dst_ip']}")
    if pd.notna(r.get("top_dst_port", np.nan)):
        parts.append(f"top_port={int(r['top_dst_port'])}")
    if r.get("top_transport", ""):
        parts.append(f"transport={r['top_transport']}")
    if r.get("top_protocol", "") and r["top_protocol"] != "-":
        parts.append(f"proto={r['top_protocol']}")
    parts.append(f"raw_out={int(r.get('raw_out_bytes_sum',0))}")
    parts.append(f"raw_in={int(r.get('raw_in_bytes_sum',0))}")
    parts.append(f"succ={r.get('raw_success_rate',0):.2f}")
    return " | ".join(parts)

ctx["alert_context"] = ctx.apply(build_reason, axis=1)

print("✅ Context table:", ctx.shape)
display(ctx.head(10))

# ---- 5) Merge back into final output
out_enriched = out.merge(ctx, on=["source.ip", "window_start"], how="left")

# Fill missing context (should only be missing if raw merge missed some)
fill_cols = [
    "raw_rows","top_dst_ip","top_dst_port","top_transport","top_protocol",
    "raw_bytes_sum","raw_pkts_sum","raw_out_bytes_sum","raw_in_bytes_sum",
    "raw_success_rate","raw_fail_rate","raw_out_in_ratio","alert_context"
]
for c in fill_cols:
    if c in out_enriched.columns:
        if out_enriched[c].dtype == object:
            out_enriched[c] = out_enriched[c].fillna("")
        else:
            out_enriched[c] = out_enriched[c].fillna(0)

# ---- 6) Show top alerts with context
display(
    out_enriched.loc[out_enriched["severity"] > 0, [
        "source.ip","window_start","severity","alert_type",
        "ae_score","lstm_score",
        "top_dst_ip","top_dst_port","top_transport","top_protocol",
        "raw_out_bytes_sum","raw_in_bytes_sum","raw_success_rate",
        "beacon_best_dst","alert_context"
    ]]
    .sort_values("severity", ascending=False)
    .head(30)
)

# ---- 7) Export enriched CSV
OUT_CSV2 = "conn_ml_scored_severity_enriched.csv"
out_enriched.to_csv(OUT_CSV2, index=False)
print("📦 Saved:", OUT_CSV2)

from google.colab import files
files.download(OUT_CSV2)


✅ Alerts to enrich: 465
✅ Raw rows for alerts: 86586
✅ Context table: (465, 20)


,source.ip,window_start,raw_rows,top_dst_ip,top_dst_port,top_transport,top_protocol,raw_bytes_sum,raw_pkts_sum,raw_out_bytes_sum,raw_in_bytes_sum,raw_out_sr_sum,raw_in_sr_sum,sf_count,rej_count,fail_count,raw_success_rate,raw_fail_rate,raw_out_in_ratio,alert_context
0,10.10.10.18,2025-10-26 13:18:00,32,8.8.8.8,80.0,tcp,-,706418312.0,931632.0,36565068.0,669853244.0,8652.0,24644.0,8,10,14,0.250000,0.437500,0.054587,top_dst=8.8.8.8 | top_port=80 | transport=tcp ...
1,10.10.10.18,2025-11-02 11:50:00,167,67.159.25.2,80.0,tcp,-,2219742.0,8170.0,1140610.0,1079132.0,580116.0,760076.0,57,3,20,0.341317,0.119760,1.056970,top_dst=67.159.25.2 | top_port=80 | transport=...
2,10.10.10.18,2025-11-04 12:03:00,204,10.10.34.35,443.0,tcp,-,2646590.0,5822.0,1156838.0,1489752.0,453134.0,1079602.0,53,62,127,0.259804,0.622549,0.776531,top_dst=10.10.34.35 | top_port=443 | transport...
3,10.10.10.18,2025-11-04 12:35:00,132,8.8.8.8,443.0,tcp,-,1464926.0,4388.0,961328.0,503598.0,495078.0,339140.0,51,15,53,0.386364,0.401515,1.908919,top_dst=8.8.8.8 | top_port=443 | transport=tcp...
4,10.10.10.18,2025-11-04 12:36:00,230,10.10.34.35,443.0,tcp,-,3533662.0,7016.0,2602970.0,930692.0,1910702.0,690670.0,46,39,166,0.200000,0.721739,2.796811,top_dst=10.10.34.35 | top_port=443 | transport...
5,10.10.10.18,2025-11-04 12:43:00,206,10.10.34.35,443.0,tcp,-,2094720.0,5350.0,1097136.0,997584.0,388228.0,564672.0,70,62,108,0.339806,0.524272,1.099793,top_dst=10.10.34.35 | top_port=443 | transport...
6,10.10.10.18,2025-11-04 12:48:00,62,67.159.25.2,80.0,tcp,http,37923559.0,38059.0,37106751.0,816808.0,37086159.0,792456.0,46,0,13,0.741935,0.209677,45.428976,top_dst=67.159.25.2 | top_port=80 | transport=...
7,10.10.10.18,2025-11-06 09:58:00,99,8.8.8.8,443.0,tcp,-,1198266.0,2832.0,571336.0,626930.0,329348.0,552832.0,38,8,48,0.383838,0.484848,0.911323,top_dst=8.8.8.8 | top_port=443 | transport=tcp...
8,10.10.10.18,2025-11-06 10:14:00,21,8.8.8.8,443.0,udp,-,91580.0,292.0,24764.0,66816.0,11260.0,22810.0,10,6,8,0.476190,0.380952,0.370630,top_dst=8.8.8.8 | top_port=443 | transport=udp...
9,10.10.10.18,2025-11-06 10:17:00,264,10.10.34.35,443.0,tcp,-,884224.0,2432.0,558952.0,325272.0,193792.0,186610.0,27,51,204,0.102273,0.772727,1.718414,top_dst=10.10.34.35 | top_port=443 | transport...


,source.ip,window_start,severity,alert_type,ae_score,lstm_score,top_dst_ip,top_dst_port,top_transport,top_protocol,raw_out_bytes_sum,raw_in_bytes_sum,raw_success_rate,beacon_best_dst,alert_context
84504,192.168.20.33,2025-11-04 12:27:00,98,BEACONING,222.114716,2.191599,192.168.100.5,443.0,tcp,-,854225.0,4.031517e+06,0.222222,23.46.189.152,top_dst=192.168.100.5 | top_port=443 | transpo...
84498,192.168.20.33,2025-11-04 12:21:00,98,BEACONING,163.812424,2.932275,192.168.100.5,389.0,tcp,-,61004.0,8.385070e+05,0.333333,23.46.189.152,top_dst=192.168.100.5 | top_port=389 | transpo...
79728,192.168.20.15,2025-11-04 11:50:00,98,BEACONING,557.492065,0.662395,192.168.100.33,8383.0,tcp,-,21824.0,2.414228e+06,0.370558,172.217.18.14,top_dst=192.168.100.33 | top_port=8383 | trans...
45927,192.168.100.28,2025-11-07 12:32:00,98,BEACONING,351.800446,2.948296,192.168.100.3,443.0,tcp,-,3258.0,5.304352e+11,0.528736,199.232.210.172,top_dst=192.168.100.3 | top_port=443 | transpo...
76901,192.168.20.13,2025-11-04 12:20:00,98,BEACONING,108.590141,2.294149,172.16.60.2,8080.0,tcp,http,131778.0,1.477933e+06,0.678363,216.58.206.35,top_dst=172.16.60.2 | top_port=8080 | transpor...
16372,192.168.10.15,2025-11-02 12:27:00,98,SLOW_EXFIL,0.780634,21.580244,35.190.80.1,443.0,tcp,ssl,177402.0,2.817440e+05,0.800000,96.17.207.137,top_dst=35.190.80.1 | top_port=443 | transport...
72774,192.168.20.10,2025-10-26 13:50:00,98,SLOW_EXFIL,4.293334,65.826469,192.168.100.5,21116.0,tcp,-,64104.0,1.388160e+05,0.652174,37.32.10.75,top_dst=192.168.100.5 | top_port=21116 | trans...
72744,192.168.20.10,2025-10-26 13:20:00,98,SLOW_EXFIL,26.965891,19.840752,192.168.100.5,53.0,udp,dns,67125256.0,2.721959e+08,0.666667,91.241.20.78,top_dst=192.168.100.5 | top_port=53 | transpor...
253,10.10.10.18,2025-11-10 09:51:00,98,BEACONING,5.615016,21.141642,8.8.8.8,443.0,udp,-,56728.0,6.000000e+02,0.125000,157.240.30.142,top_dst=8.8.8.8 | top_port=443 | transport=udp...
1718,10.10.10.200,2025-11-10 09:57:00,98,SLOW_EXFIL,2.116078,19.415586,224.0.0.251,0.0,icmp,-,18408.0,3.283200e+04,0.000000,10.10.10.1,top_dst=224.0.0.251 | top_port=0 | transport=i...


📦 Saved: conn_ml_scored_severity_enriched.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>